1. Importar librerías

In [ ]:
# ==========================================
# 1. Importar librerías
# ==========================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 6)


2. Cargar dataset limpio

In [ ]:
# ==========================================
# 2. Cargar dataset limpio
# ==========================================
import os
project_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
file_path = os.path.join(project_root, "data/processed/online_news_cleaned.csv")

df = pd.read_csv(file_path)
df.head()



# 🟥 Grupo F — Variables del propio artículo

**Variables:**
- `url`
- `timedelta`
- `self_reference_*`

**👉 Aquí analizas:**
- antigüedad del artículo  
- si se referencia a sí mismo (posible señal de autoridad)  
- número de enlaces internos  

Tabla explicativa de variables (Markdown)

| Variable | Qué representa | Tipo | Interpretación |
|---------|----------------|------|----------------|
| `url` | Dirección web donde se publicó el artículo | Categórica | Permite extraer patrones: dominio, sección, fecha. No se usa directamente como numérica, pero puede generar nuevas features. |
| `timedelta` | Días entre publicación y fecha de extracción | Numérica | Mide antigüedad. Los artículos recientes tienden a ser más virales. |
| `num_self_hrefs` | Número de enlaces internos hacia otros artículos del mismo sitio | Numérica | Señal del nivel de interconexión interna; puede aumentar tráfico. |
| `self_reference_min_shares` | Shares mínimos de los artículos propios referenciados | Numérica | Si el artículo enlaza contenido poco popular, es señal de baja autoridad interna. |
| `self_reference_max_shares` | Shares máximos de artículos propios referenciados | Numérica | Si enlaza contenido muy viral: señal de autoridad y relevancia. |
| `self_reference_avg_sharess` | Promedio de shares de los artículos propios referenciados | Numérica | Indica autoridad interna promedio del contenido vinculado. |


Interpretación del grupo F (qué aporta al modelo)

**Interpretación de valor para el modelo:**

- `timedelta`: relevante para medir frescura del contenido. Artículos más viejos suelen tener menos shares → feature útil.
- Variables de self-reference: aportan señales de *autoridad interna* del artículo y del medio.  
  - Si un artículo enlaza otros artículos que fueron muy virales → señal de experticia.  
  - Si enlaza artículos con shares bajos → podría indicar temas poco relevantes.  
- Pueden ayudar a captar efectos de red interna del sitio, por ejemplo:  
  - contenido evergreen,  
  - artículos de pilar,  
  - enlaces internos estratégicos.


1️⃣ Definir variables del grupo

In [ ]:
# ==============================
# 1. Definir variables del Grupo F
# ==============================

vars_F = [
    "url",
    "timedelta",
    "num_self_hrefs",
    "self_reference_min_shares",
    "self_reference_max_shares",
    "self_reference_avg_sharess"
]
df[vars_F].head()


2️⃣ Estadísticas descriptivas de variables numéricas

In [ ]:
# ==============================
# 2. Estadísticas descriptivas
# ==============================

numeric_F = [
    "timedelta",
    "num_self_hrefs",
    "self_reference_min_shares",
    "self_reference_max_shares",
    "self_reference_avg_sharess"
]

df[numeric_F].describe().T


🟦 5. Distribución de antigüedad (timedelta)

In [ ]:
# ==========================================
# 5. Distribución de timedelta
# ==========================================

plt.figure(figsize=(10,5))
sns.histplot(df["timedelta"], bins=40, kde=True)
plt.title("Distribución de la antigüedad del artículo (timedelta)")
plt.xlabel("Días desde publicación")
plt.ylabel("Frecuencia")
plt.show()


🟦 6. Distribución log-scale de variables sesgadas

In [ ]:
# =========================
# 8. Comparación antes/después de log-transform
# =========================

for col in numeric_F[1:]:
    fig, axes = plt.subplots(1, 2, figsize=(12,4))

    sns.histplot(df[col], bins=30, ax=axes[0])
    axes[0].set_title(f"{col} — Original")

    sns.histplot(np.log1p(df[col]), bins=30, ax=axes[1], color="orange")
    axes[1].set_title(f"{col} — Log Transform")

    plt.show()


🟦 7. Scatterplots log-log vs. shares

In [ ]:
# ==========================================
# 7. Relación con shares (log-log)
# ==========================================

for col in numeric_F:
    plt.figure(figsize=(8,5))
    sns.scatterplot(x=df[col], y=df["shares"], alpha=0.4)
    plt.xscale("log")
    plt.yscale("log")
    plt.title(f"{col} vs shares (log-log)")
    plt.xlabel(col)
    plt.ylabel("shares")
    plt.show()


🟦 8. Matriz de correlación

In [ ]:
# ==========================================
# 8. Matriz de correlación
# ==========================================

plt.figure(figsize=(8,6))
sns.heatmap(df[numeric_F + ["shares"]].corr(), annot=True, cmap="coolwarm")
plt.title("Correlación Grupo F con shares")
plt.show()


🧩 SECCIÓN 1 — Calidad de datos (Missing Values)
📌 ¿Por qué es importante?

En MLOps es obligatorio validar si el dataset tiene valores nulos porque:

Afectan las transformaciones del pipeline

Pueden causar fallos en producción

Son indicadores tempranos de data drift

In [ ]:
# =========================
# 1. Calidad de datos (Missing Values)
# =========================

df[vars_F].isnull().sum().to_frame("missing_values")


🧩 SECCIÓN 3 — Transformaciones recomendadas (log-transform)
📌 ¿Por qué es importante?

En MLOps necesitas justificar por qué transformas variables, y la razón es:

Reducir skewness

Estabilizar varianza

Mejorar desempeño de modelos lineales

Evitar explosiones numéricas

In [ ]:
# =========================
# 3. Transformaciones sugeridas
# =========================

transformations = {
    "log_transform": [
        "num_self_hrefs",
        "self_reference_min_shares",
        "self_reference_max_shares",
        "self_reference_avg_sharess"
    ],
    "no_transform": ["timedelta"],
    "scaling": ["timedelta", "num_self_hrefs"],
    "imputation": "median"
}

transformations


🧩 SECCIÓN 4 — Detección de outliers (IQR)
📌 ¿Por qué es importante?

Los outliers afectan:

Estabilidad de modelos

Distribuciones futuras

Sensibilidad en inferencia

Monitoreo de producción (data drift)

In [ ]:
# =========================
# 4. Detección de outliers
# =========================

for col in numeric_F:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    upper = q3 + 1.5 * iqr
    lower = q1 - 1.5 * iqr
    outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    print(f"{col}: {outliers} outliers")


🧩 SECCIÓN 5 — Análisis de colinealidad (VIF)
📌 ¿Por qué es importante?

En pipelines de MLOps:

Alta colinealidad causa inestabilidad

Afecta interpretabilidad

Rompe modelos lineales

In [ ]:
# =========================
# 5. Colinealidad (VIF)
# =========================

from statsmodels.stats.outliers_influence import variance_inflation_factor

X = df[numeric_F].fillna(0)
vif = pd.DataFrame()
vif["feature"] = X.columns
vif["VIF"] = [variance_inflation_factor(X.values, i)
              for i in range(len(X.columns))]
vif


🧩 SECCIÓN 6 — Feature Engineering recomendado
📌 ¿Por qué es importante?

Permite:

Mejorar el rendimiento del modelo

Resumir patrones complejos

Crear features robustos a drift

🚀 Nuevos features propuestos:

authority_score → autoridad interna del artículo

internal_links_log → log de enlaces internos

is_old_article → flag binario para antigüedad

In [ ]:
# =========================
# 6. Feature Engineering
# =========================

df["authority_score"] = np.log1p(df["self_reference_avg_sharess"])
df["internal_links_log"] = np.log1p(df["num_self_hrefs"])
df["is_old_article"] = (df["timedelta"] > 400).astype(int)

df[["authority_score", "internal_links_log", "is_old_article"]].head()


🧩 SECCIÓN 7 — Verificación de Leakage
📌 ¿Por qué es importante?

Antes de modelar, debes asegurar que ninguna feature contiene información del futuro, lo cual:

Rompería el modelo

Haría imposible el monitoreo en producción

Es un error crítico en MLOps

🚨 Se buscan columnas que contengan "shares".

In [ ]:
# =========================
# 7. Verificación de leakage
# =========================

potential_leakage = [
    col for col in vars_F if "shares" in col and col != "shares"
]

potential_leakage


Crear nuevas features de interacción

In [ ]:
# ================================
# 1. Nuevas interacciones propuestas
# ================================

# Interacción: autoridad × antigüedad
df["authority_x_timedelta"] = df["authority_score"] * df["timedelta"]

# Interacción: enlaces internos × categoría (día/canales)
# Aquí usamos num_self_hrefs multiplicado por dummies
data_channel_cols = [col for col in df.columns if col.startswith("data_channel_is_")]
weekday_cols = [col for col in df.columns if col.startswith("weekday_is_")]

# Crear interacciones canal × enlaces internos
for col in data_channel_cols:
    df[f"links_x_{col}"] = df["num_self_hrefs"] * df[col]

# Crear interacciones día × enlaces internos
for col in weekday_cols:
    df[f"links_x_{col}"] = df["num_self_hrefs"] * df[col]

df[["authority_x_timedelta"] + [c for c in df.columns if "links_x_" in c]].head()


Evaluar si estas interacciones aportan valor

In [ ]:
# ================================
# 2. Correlación simple con shares
# ================================

interaction_cols = ["authority_x_timedelta"] + [c for c in df.columns if "links_x_" in c]

corrs = df[interaction_cols + ["shares"]].corr()["shares"].sort_values(ascending=False)
corrs.head(10)


Importancia de features con un modelo rápido (RandomForest)

In [ ]:
# ================================
# 3. Importancia de features
# ================================
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

X = df[interaction_cols]
y = df["shares"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rf = RandomForestRegressor(n_estimators=300, random_state=42)
rf.fit(X_train, y_train)

importances = pd.DataFrame({
    "feature": X.columns,
    "importance": rf.feature_importances_
}).sort_values("importance", ascending=False)

importances.head(10)


Comparar modelo base vs modelo con interacciones

In [ ]:
base_features = ["authority_score", "num_self_hrefs", "timedelta"]

X_base = df[base_features]
X_full = df[base_features + interaction_cols]

from sklearn.metrics import r2_score

X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(X_base, y, test_size=0.2, random_state=42)
X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(X_full, y, test_size=0.2, random_state=42)

rf_base = RandomForestRegressor(n_estimators=300, random_state=42)
rf_full = RandomForestRegressor(n_estimators=300, random_state=42)

rf_base.fit(X_train_b, y_train_b)
rf_full.fit(X_train_f, y_train_f)

r2_base = r2_score(y_test_b, rf_base.predict(X_test_b))
r2_full = r2_score(y_test_f, rf_full.predict(X_test_f))

r2_base, r2_full


# ✅ **Principales descubrimientos del Grupo F (Análisis Detallado)**

## 1️⃣ Las variables de *self-reference* presentan un sesgo extremo  
Las variables:
- `self_reference_min_shares`
- `self_reference_max_shares`
- `self_reference_avg_sharess`

presentan distribuciones altamente asimétricas (right-skewed).  
Se observan valores máximos superiores a **80,000–110,000 shares**, típicos de fenómenos de viralidad.

**Por qué importa:**  
- Los modelos lineales fallan con distribuciones tan sesgadas.  
- La alta varianza impacta estabilidad del entrenamiento.  
- Se requiere aplicar `log1p()` para estabilizar la escala.

---

## 2️⃣ Requieren log-transform para estabilizar la distribución  
La transformación `np.log1p()`:

- reduce skewness,  
- hace las distribuciones más simétricas,  
- y mejora la relación con `shares` bajo escala log-log.

**Impacto en el modelo:**  
- mejora estabilidad numérica,  
- reduce efecto de outliers,  
- favorece modelos lineales y boosting.

---

## 3️⃣ La “autoridad interna” del artículo es un predictor relevante  
Las variables de self-reference capturan si el artículo enlaza contenidos:

- altamente virales (alta autoridad),  
- o poco relevantes (baja autoridad).

**Conclusión:**  
Artículos que enlazan contenido viral tienden a recibir más shares.

---

## 4️⃣ Los artículos que enlazan contenido viral tienden a recibir más shares  
La matriz de correlación confirma que:

- `self_reference_avg_sharess` es la más asociada con `shares`.

Esto respalda la hipótesis de transmisión interna de viralidad en el sitio.

---

## 5️⃣ La variable `timedelta` muestra que la antigüedad afecta la viralidad  
- Artículos recientes tienen más interacción.  
- Artículos viejos acumulan menos tráfico.

**Interpretación:**  
La frescura del contenido es un factor importante → se debe preservar y escalar.

---

## 6️⃣ Existen miles de outliers, pero son parte natural del fenómeno  
Se detectaron:
- **2,000–5,000 outliers** por variable.

**Por qué no eliminarlos:**  
- Representan eventos reales de viralidad.  
- Eliminarlos distorsiona la distribución.  
- Modelos como RandomForest y XGBoost los manejan correctamente.

---

## 7️⃣ No hay valores nulos (Missing Values = 0)  
Esto simplifica:

- el pipeline,  
- la integración con modelos,  
- la implementación en MLOps.

No se requiere imputación.

---

## 8️⃣ El VIF es aceptable → no hay multicolinealidad severa  
Todos los valores de VIF < 5, indicando:

- baja colinealidad,  
- estabilidad en el modelado,  
- ninguna variable debe eliminarse por redundancia.

---

## 9️⃣ Las relaciones no lineales se observan claramente en log-log  
Bajo escala log-log:

- las relaciones se vuelven más lineales,  
- se reduce heterocedasticidad,  
- mejora la interpretabilidad.

Esto valida el uso de transformaciones logarítmicas.

---

# 🚀 **Recomendaciones actualizadas para el pipeline MLOps**

## 1️⃣ Incluir log-transform en el pipeline  
Aplicar `np.log1p()` a:
- `self_reference_min_shares`
- `self_reference_max_shares`
- `self_reference_avg_sharess`
- `num_self_hrefs` (recomendado)

**Razón:** reduce skewness y estabiliza distribuciones antes del modelado.

---

## 2️⃣ Escalar `timedelta` y `num_self_hrefs`  
Aplicar escalado dentro del Pipeline:

- `StandardScaler` para modelos lineales,  
- `MinMaxScaler` para modelos basados en distancias.

---

## 3️⃣ Mantener los outliers  
No eliminarlos.  
Se conservarán en el pipeline porque:

- reflejan comportamiento real,  
- modelos basados en árboles los manejan bien.

---

## 4️⃣ ❗ No agregar interacciones nuevas (conclusión basada en evidencia)  
Se evaluaron las interacciones:

- `authority_x_timedelta`  
- `links_x_*_category`  

Y se encontró que:

- su correlación con shares es muy baja (máx ≈ 0.04),  
- no mejoran el R² del modelo,  
- añaden ruido y complejidad innecesaria.

**Conclusión:**  
➡️ **No incluir estas interacciones en el pipeline.**

---

## 5️⃣ Versionar todas las transformaciones  
Guardar en MLflow:

- scalers,  
- pipelines de log-transform,  
- codificadores categóricos.

**Propósito:** garantizar trazabilidad en producción.

---

## 6️⃣ Vigilar potencial leakage  
Documentar la procedencia temporal de:

- `self_reference_min_shares`,  
- `self_reference_max_shares`,  
- `self_reference_avg_sharess`.

Asegurar que el contenido referenciado existía antes del artículo actual.

---

## 7️⃣ Incorporar nuevas features confirmadas  
Estas sí han mostrado utilidad:

- `authority_score = log(self_reference_avg_sharess)`  
- `internal_links_log = log(num_self_hrefs)`  
- `is_old_article = int(timedelta > 400)`

Añaden robustez y capturan patrones relevantes.

---

## 8️⃣ Preparar validación temporal (Time-Based Split)  
Evitar dividir aleatoriamente.  
Usar:

- `TimeSeriesSplit`,  
- o separación manual por fecha.

Esto evita leakage temporal y simula un flujo real de producción.

---
